# Track work field changes (oxjob #2539, step 1: shadow)

**What it answers.** Each night the End 2 End gives tens of millions of works a new `updated_date` (68M on 22 Nov 2026),
and a work's `updated_date` says only "something changed". Most of those works changed one thing (72-87% changed exactly
one field group, 18-22 Nov). This notebook records, per work, **which served fields changed at each stamp**, so we can
offer syncers "the works where these fields changed since D" (oxjob #2539: a nightly changes file and an `updated_fields`
API filter, their shape still to be approved). Step 1 is internal only: nothing reads these tables yet.

**How.** For every work whose published `updated_date` moved since this notebook last ran (plus new and deleted works),
it hashes each tracked field's served value (`xxhash64`, one per field) and compares with the hashes stored at the work's
previous stamp. The fields that differ take the work's new `updated_date` as their field date. A stamped work with no
tracked difference records `other` (a forced re-stamp, or a change only in an unserved column such as `publisher`).

Why it reads the published `openalex_works`, not `openalex_works_hash`: a field date can then move only when the served
`updated_date` moves, so every field date is a stamp of that work and the invariant "updated_date = latest field date
or `other`" holds by construction. Rebaselines (hash changed, date kept: the night-wide flag or #1322's per-row
precompute) and forced re-stamps (stored hash flipped) need no special case: a rebaselined change is reported at the
work's next stamp, which is when a date syncer receives it too (the trickle the walden rule describes). Diffing against
the work's last stamp, not last night, is what makes that right. It reads only accepted nights (after Publish_Works).

**Tracked names** are the work API's served top-level fields (the names `select` takes), with `display_name` = `title`,
plus `authorships.author` and `authorships.institutions` (affiliations, the third most-asked need), and two search-only
pseudo-fields, `abstract` and `fulltext`. Derived counts (`locations_count`, `referenced_works_count`,
`countries_distinct_count`, `institutions_distinct_count`, `is_authors_truncated`) move with their base field.
Locations are hashed on their served projection (LocationSchema's 13 fields), so internal keys such as `url_sort_score`
or `endpoint_id` never count as a change. `fwci`, `citation_normalized_percentile` (its value) and
`cited_by_percentile_year` are recomputed nightly for 55-81M works without a stamp, so they are diffed separately, every
night over every work (last cells); the top-10%/1% flags, which do stamp, are tracked as `citation_normalized_percentile`.
Adding a name later is safe: a work whose stored hashes lack the name takes it silently at its next stamp, never as a change.
**Changing a name's hash expression: give its key a new version** (`name@2`, `name@3`): the old key then counts as absent,
so the change of expression is never reported as a change of value; the logs and `field_dates` use the name without
`@N`. (Editing an expression in place would report every work whose hash moves, at its next stamp.) The cost: on the
night a key's version changes, that field's changes are not recorded (every stored hash lacks the new key).
Hashing note: `xxhash64` treats NULL and `[]` alike and skips them, which is wanted (`[]` vs NULL is no change to a
reader), but it also drops array boundaries, so arrays of arrays carry each element's index (`authorships.*`).
`institutions` (top level) is not tracked: it is empty in `openalex_works` (22 Nov 2026).

**Tables** (all in `:target_schema`, `openalex.works` in production):
- `openalex_works_field_state`: one row per work: `updated_date` (its last stamp seen here), `field_hashes`,
  `field_dates` (per field, the last stamp at which it changed; empty until it does), `changed_fields` and `change`
  of the last stamp.
- `openalex_works_field_changes`: the nightly log, partitioned by `night` (the UTC date of the night's stamps): one row
  per work stamped (`new` / `updated`, with `changed_fields`) or deleted.
- `openalex_works_metric_state` / `openalex_works_metric_changes`: the same for the three nightly metrics.
- `openalex_works_field_change_counts`: per night, works per change and per field (the shadow week's numbers).
- `openalex_works_field_runs`: one row per night: watermark, bootstrap flag, timings, consistency counts.

**First run = bootstrap.** With an empty state it stores every work's hashes and logs nothing: there is no history
before launch. A lost state table comes back the same way (a gap, never false changes).

**Idempotent.** A re-run of the same night (a repair, or the trigger firing twice) changes nothing: the run row keeps
the night's watermark, the MERGE skips works whose stored date already equals the published one, and the logs are
rewritten for the night.

**Revert:** pause or delete the job (`jobs/work_field_changes.yaml`) and drop the six tables; nothing reads them.

### Tables

In [ ]:
-- oxjob #2539: created once; every statement below names the schema through :target_schema (openalex.works in
-- production, a scratch schema for replays of past nights).
CREATE TABLE IF NOT EXISTS IDENTIFIER(:target_schema || '.openalex_works_field_state') (
  id BIGINT NOT NULL,
  updated_date TIMESTAMP COMMENT 'the work''s updated_date at its last stamp this table saw',
  field_hashes MAP<STRING, BIGINT> COMMENT 'xxhash64 of each tracked field''s served value at updated_date',
  field_dates MAP<STRING, TIMESTAMP> COMMENT 'per tracked field, the last stamp at which its served value changed (since tracking began)',
  changed_fields ARRAY<STRING> COMMENT 'what changed at updated_date: field names, or other',
  change STRING COMMENT 'bootstrap | new | updated'
)
COMMENT 'oxjob #2539: per-field change dates of works, derived from the published updated_date. Written by notebooks/end2end/TrackWorkFieldChanges.'
TBLPROPERTIES ('delta.enableDeletionVectors' = 'true', 'delta.deletedFileRetentionDuration' = '7 days', 'delta.logRetentionDuration' = '7 days');

CREATE TABLE IF NOT EXISTS IDENTIFIER(:target_schema || '.openalex_works_field_changes') (
  night DATE NOT NULL COMMENT 'UTC date of the night''s stamps (the daily snapshot''s updated_date partition)',
  id BIGINT NOT NULL,
  updated_date TIMESTAMP COMMENT 'the work''s new updated_date (NULL for deleted)',
  change STRING NOT NULL COMMENT 'new | updated | deleted',
  changed_fields ARRAY<STRING> COMMENT 'tracked fields that changed; every field for new; [other] when none did'
)
PARTITIONED BY (night)
COMMENT 'oxjob #2539: the nightly per-field change log of works. Written by notebooks/end2end/TrackWorkFieldChanges.';

CREATE TABLE IF NOT EXISTS IDENTIFIER(:target_schema || '.openalex_works_metric_state') (
  night DATE,
  id BIGINT NOT NULL,
  fwci DOUBLE,
  citation_normalized_percentile STRUCT<value: DOUBLE, is_in_top_1_percent: BOOLEAN, is_in_top_10_percent: BOOLEAN>,
  cited_by_percentile_year STRUCT<min: INT, max: INT>
)
COMMENT 'oxjob #2539: last night''s nightly-recomputed metrics of every work, to diff tonight''s against.';

CREATE TABLE IF NOT EXISTS IDENTIFIER(:target_schema || '.openalex_works_metric_changes') (
  night DATE NOT NULL,
  id BIGINT NOT NULL,
  changed_fields ARRAY<STRING> NOT NULL COMMENT 'fwci | citation_normalized_percentile | cited_by_percentile_year'
)
PARTITIONED BY (night)
COMMENT 'oxjob #2539: works whose nightly-recomputed metrics changed, per night (no updated_date move).';

CREATE TABLE IF NOT EXISTS IDENTIFIER(:target_schema || '.openalex_works_field_change_counts') (
  night DATE NOT NULL,
  change STRING NOT NULL COMMENT 'new | updated | deleted | metric',
  field STRING COMMENT 'NULL = all works of this change',
  works BIGINT NOT NULL
)
COMMENT 'oxjob #2539: per night, works per change and per changed field.';

CREATE TABLE IF NOT EXISTS IDENTIFIER(:target_schema || '.openalex_works_field_runs') (
  night DATE NOT NULL,
  works_table STRING,
  watermark TIMESTAMP COMMENT 'max(updated_date) of the state before this night: works stamped after it are processed',
  bootstrap BOOLEAN,
  started_at TIMESTAMP,
  finished_at TIMESTAMP,
  works BIGINT,
  works_missing_from_state BIGINT COMMENT 'should be 0',
  state_rows_not_in_works BIGINT COMMENT 'should be 0',
  date_mismatches BIGINT COMMENT 'works whose published updated_date differs from the stored one after the run: an out-of-band date write below the watermark; should be 0',
  duplicate_ids BIGINT COMMENT 'should be 0'
)
COMMENT 'oxjob #2539: one row per night TrackWorkFieldChanges ran.';

### Tonight's run row

`night` is the UTC date of the newest stamp in the published table, so the trigger firing twice on one night, or a
repair, reuses the row and its watermark. Guard: the published table must hold at least 90% of the works the state
knows (a broken read would otherwise log most works as deleted).

In [ ]:
INSERT INTO IDENTIFIER(:target_schema || '.openalex_works_field_runs') (night, works_table, watermark, bootstrap, started_at)
SELECT n.night, :works_table, st.watermark, st.watermark IS NULL, current_timestamp()
FROM (SELECT to_date(max(updated_date)) AS night FROM IDENTIFIER(:works_table)) n
CROSS JOIN (SELECT max(updated_date) AS watermark FROM IDENTIFIER(:target_schema || '.openalex_works_field_state')) st
WHERE NOT EXISTS (SELECT 1 FROM IDENTIFIER(:target_schema || '.openalex_works_field_runs') r WHERE r.night = n.night);

SELECT CASE
  WHEN (SELECT count(*) FROM IDENTIFIER(:target_schema || '.openalex_works_field_state')) > 0
   AND (SELECT count(*) FROM IDENTIFIER(:works_table)) < 0.9 * (SELECT count(*) FROM IDENTIFIER(:target_schema || '.openalex_works_field_state'))
  THEN RAISE_ERROR(CONCAT('WORK FIELD CHANGES GUARD: ', :works_table, ' has fewer than 90% of the works the state holds. Nothing was written.'))
END;

### Stamped works: per-field hashes and dates

One MERGE. Source: every work stamped after the watermark (on the bootstrap night, every work). Matched works whose
stored date differs take tonight's hashes; the fields whose hash differs take the work's new `updated_date` in
`field_dates`; none differing records `other`. A name the stored hashes lack (added to the list later) is never a
change. New works get every field dated; bootstrap rows get none.

The field list below is the one place tracked names are defined; a key's `@N` suffix versions its expression.

In [ ]:
MERGE INTO IDENTIFIER(:target_schema || '.openalex_works_field_state') AS t
USING (
  SELECT
    w.id,
    w.updated_date,
    r.bootstrap,
    map(
      'doi', xxhash64(w.doi),
      'ids', xxhash64(to_json(w.ids)),
      'title', xxhash64(w.title, w.title_abstract_number),
      'alternate_titles', xxhash64(w.alternate_titles),
      'publication_date', xxhash64(w.publication_date),
      'publication_year', xxhash64(year(w.publication_date)),
      'language', xxhash64(w.language),
      'type', xxhash64(w.type),
      'type_crossref', xxhash64(w.type_crossref),
      'abstract_inverted_index', xxhash64(w.abstract_inverted_index),
      'abstract', xxhash64(w.abstract),
      'fulltext', xxhash64(w.fulltext),
      'has_fulltext', xxhash64(w.fulltext IS NOT NULL),
      'has_content', xxhash64(w.has_content),
      'authorships', xxhash64(w.authorships),
      'authorships.author', xxhash64(transform(w.authorships, (a, i) -> struct(i, a.author))),
      'authorships.institutions', xxhash64(transform(w.authorships, (a, i) -> struct(i, a.institutions))),
      'corresponding_author_ids', xxhash64(w.corresponding_author_ids),
      'corresponding_institution_ids', xxhash64(w.corresponding_institution_ids),
      'topics', xxhash64(transform(w.topics, t -> struct(t.id, t.display_name, round(t.score, 4), t.subfield, t.field, t.domain))),
      'primary_topic', xxhash64(w.primary_topic.id, w.primary_topic.display_name, round(w.primary_topic.score, 4), w.primary_topic.subfield, w.primary_topic.field, w.primary_topic.domain),
      'keywords', xxhash64(transform(w.keywords, k -> struct(k.id, k.display_name, round(k.score, 4)))),
      'concepts', xxhash64(transform(w.concepts, c -> struct(c.id, c.wikidata, c.display_name, c.level, round(c.score, 4)))),
      'sustainable_development_goals', xxhash64(transform(w.sustainable_development_goals, s -> struct(s.id, s.display_name, round(s.score, 4)))),
      'study_designs', xxhash64(w.study_designs),
      'mesh', xxhash64(w.mesh),
      'referenced_works', xxhash64(w.referenced_works),
      'related_works', xxhash64(w.related_works),
      'pull_notice_works', xxhash64(w.pull_notice_works),
      'cited_by_count', xxhash64(w.cited_by_count),
      'counts_by_year', xxhash64(w.counts_by_year),
      'reviewed_by_count', xxhash64(w.reviewed_by_count),
      'citation_normalized_percentile', xxhash64(coalesce(w.citation_normalized_percentile.is_in_top_1_percent, false), coalesce(w.citation_normalized_percentile.is_in_top_10_percent, false)),
      'locations', xxhash64(transform(w.locations, l -> named_struct('id', concat(l.provenance, ':', l.native_id), 'doi', l.doi, 'is_oa', l.is_oa, 'landing_page_url', l.landing_page_url, 'pdf_url', l.pdf_url, 'source', l.source, 'license', l.license, 'license_id', l.license_id, 'version', l.version, 'is_accepted', l.is_accepted, 'raw_source_name', l.raw_source_name, 'raw_type', l.raw_type))),
      'primary_location', xxhash64(named_struct('id', concat(w.primary_location.provenance, ':', w.primary_location.native_id), 'doi', w.primary_location.doi, 'is_oa', w.primary_location.is_oa, 'landing_page_url', w.primary_location.landing_page_url, 'pdf_url', w.primary_location.pdf_url, 'source', w.primary_location.source, 'license', w.primary_location.license, 'license_id', w.primary_location.license_id, 'version', w.primary_location.version, 'is_accepted', w.primary_location.is_accepted, 'is_published', w.primary_location.is_published, 'raw_source_name', w.primary_location.raw_source_name, 'raw_type', w.primary_location.raw_type)),
      'best_oa_location', xxhash64(named_struct('id', concat(w.best_oa_location.provenance, ':', w.best_oa_location.native_id), 'doi', w.best_oa_location.doi, 'is_oa', w.best_oa_location.is_oa, 'landing_page_url', w.best_oa_location.landing_page_url, 'pdf_url', w.best_oa_location.pdf_url, 'source', w.best_oa_location.source, 'license', w.best_oa_location.license, 'license_id', w.best_oa_location.license_id, 'version', w.best_oa_location.version, 'is_accepted', w.best_oa_location.is_accepted, 'is_published', w.best_oa_location.is_published, 'raw_source_name', w.best_oa_location.raw_source_name, 'raw_type', w.best_oa_location.raw_type)),
      'open_access', xxhash64(w.open_access),
      'indexed_in', xxhash64(w.indexed_in),
      'is_retracted', xxhash64(coalesce(w.is_retracted, false)),
      'is_pulled', xxhash64(coalesce(w.is_pulled, false)),
      'is_paratext', xxhash64(w.is_paratext),
      'is_xpac', xxhash64(coalesce(w.is_xpac, false)),
      'biblio', xxhash64(w.biblio),
      'apc_list', xxhash64(w.apc_list),
      'apc_paid', xxhash64(w.apc_paid),
      'funders', xxhash64(w.funders),
      'awards', xxhash64(w.awards),
      'parent_works', xxhash64(w.parent_works),
      'lineage', xxhash64(w.lineage),
      'child_works_count', xxhash64(w.child_works_count),
      'created_date', xxhash64(w.created_date)
    ) AS fh
  FROM IDENTIFIER(:works_table) w
  CROSS JOIN (SELECT bootstrap, watermark FROM IDENTIFIER(:target_schema || '.openalex_works_field_runs') WHERE night = (SELECT max(night) FROM IDENTIFIER(:target_schema || '.openalex_works_field_runs'))) r
  WHERE r.bootstrap OR w.updated_date > r.watermark
) AS s
ON t.id = s.id
WHEN MATCHED AND NOT (t.updated_date <=> s.updated_date) THEN UPDATE SET
  t.changed_fields = CASE WHEN size(transform(filter(map_keys(s.fh), k -> map_contains_key(t.field_hashes, k) AND t.field_hashes[k] <> s.fh[k]), k -> split(k, '@')[0])) = 0 THEN array('other') ELSE transform(filter(map_keys(s.fh), k -> map_contains_key(t.field_hashes, k) AND t.field_hashes[k] <> s.fh[k]), k -> split(k, '@')[0]) END,
  t.field_dates = map_concat(
    map_filter(t.field_dates, (k, v) -> NOT array_contains(transform(filter(map_keys(s.fh), k -> map_contains_key(t.field_hashes, k) AND t.field_hashes[k] <> s.fh[k]), k -> split(k, '@')[0]), k)),
    map_from_arrays(transform(filter(map_keys(s.fh), k -> map_contains_key(t.field_hashes, k) AND t.field_hashes[k] <> s.fh[k]), k -> split(k, '@')[0]), array_repeat(s.updated_date, size(transform(filter(map_keys(s.fh), k -> map_contains_key(t.field_hashes, k) AND t.field_hashes[k] <> s.fh[k]), k -> split(k, '@')[0]))))),
  t.field_hashes = s.fh,
  t.updated_date = s.updated_date,
  t.change = 'updated'
WHEN NOT MATCHED THEN INSERT (id, updated_date, field_hashes, field_dates, changed_fields, change) VALUES (
  s.id,
  s.updated_date,
  s.fh,
  CASE WHEN s.bootstrap THEN CAST(map() AS MAP<STRING, TIMESTAMP>)
       ELSE map_from_arrays(transform(map_keys(s.fh), k -> split(k, '@')[0]), array_repeat(s.updated_date, size(s.fh))) END,
  CASE WHEN s.bootstrap THEN CAST(array() AS ARRAY<STRING>) ELSE transform(map_keys(s.fh), k -> split(k, '@')[0]) END,
  CASE WHEN s.bootstrap THEN 'bootstrap' ELSE 'new' END);

### Tonight's log: stamped and deleted works

Stamped rows are re-derived from the state (rows written after the watermark), so a re-run rewrites them. Deleted
works (in the state, gone from the published table: deleted or merged away) are logged once, then dropped from the state.

In [ ]:
DELETE FROM IDENTIFIER(:target_schema || '.openalex_works_field_changes') WHERE night = (SELECT max(night) FROM IDENTIFIER(:target_schema || '.openalex_works_field_runs')) AND change IN ('new', 'updated');

INSERT INTO IDENTIFIER(:target_schema || '.openalex_works_field_changes') (night, id, updated_date, change, changed_fields)
SELECT r.night, s.id, s.updated_date, s.change, s.changed_fields
FROM IDENTIFIER(:target_schema || '.openalex_works_field_state') s
CROSS JOIN (SELECT night, watermark FROM IDENTIFIER(:target_schema || '.openalex_works_field_runs') WHERE night = (SELECT max(night) FROM IDENTIFIER(:target_schema || '.openalex_works_field_runs')) AND NOT bootstrap) r
WHERE s.updated_date > r.watermark AND s.change IN ('new', 'updated');

INSERT INTO IDENTIFIER(:target_schema || '.openalex_works_field_changes') (night, id, updated_date, change, changed_fields)
SELECT (SELECT max(night) FROM IDENTIFIER(:target_schema || '.openalex_works_field_runs')), s.id, CAST(NULL AS TIMESTAMP), 'deleted', CAST(array() AS ARRAY<STRING>)
FROM (SELECT id FROM IDENTIFIER(:target_schema || '.openalex_works_field_state')) s
LEFT ANTI JOIN (SELECT id FROM IDENTIFIER(:works_table)) w ON w.id = s.id
LEFT ANTI JOIN (SELECT id FROM IDENTIFIER(:target_schema || '.openalex_works_field_changes') WHERE night = (SELECT max(night) FROM IDENTIFIER(:target_schema || '.openalex_works_field_runs')) AND change = 'deleted') d ON d.id = s.id;

MERGE INTO IDENTIFIER(:target_schema || '.openalex_works_field_state') AS t
USING (SELECT id FROM IDENTIFIER(:target_schema || '.openalex_works_field_changes') WHERE night = (SELECT max(night) FROM IDENTIFIER(:target_schema || '.openalex_works_field_runs')) AND change = 'deleted') AS d
ON t.id = d.id
WHEN MATCHED THEN DELETE;

### Nightly metrics: `fwci`, `citation_normalized_percentile`, `cited_by_percentile_year`

Recomputed every night for every work without moving `updated_date` (55-81M works a night), so they are diffed over
every work against last night's values, not at stamps. The state carries the night it reflects, so a re-run after it was
replaced does nothing (its log rows were written before the replace). The first run stores values and logs nothing.

In [ ]:
BEGIN
  DECLARE tonight DATE;
  DECLARE state_night DATE;
  SET tonight = (SELECT max(night) FROM IDENTIFIER(:target_schema || '.openalex_works_field_runs'));
  SET state_night = (SELECT max(night) FROM IDENTIFIER(:target_schema || '.openalex_works_metric_state'));
  IF state_night IS NULL OR state_night < tonight THEN
    IF state_night IS NOT NULL THEN
      DELETE FROM IDENTIFIER(:target_schema || '.openalex_works_metric_changes') WHERE night = tonight;
      INSERT INTO IDENTIFIER(:target_schema || '.openalex_works_metric_changes') (night, id, changed_fields)
      SELECT tonight, w.id, filter(array(
          CASE WHEN NOT (w.fwci <=> m.fwci) THEN 'fwci' END,
          CASE WHEN NOT (w.citation_normalized_percentile <=> m.citation_normalized_percentile) THEN 'citation_normalized_percentile' END,
          CASE WHEN NOT (w.cited_by_percentile_year <=> m.cited_by_percentile_year) THEN 'cited_by_percentile_year' END),
        x -> x IS NOT NULL)
      FROM IDENTIFIER(:works_table) w
      JOIN IDENTIFIER(:target_schema || '.openalex_works_metric_state') m ON m.id = w.id
      WHERE NOT (w.fwci <=> m.fwci)
         OR NOT (w.citation_normalized_percentile <=> m.citation_normalized_percentile)
         OR NOT (w.cited_by_percentile_year <=> m.cited_by_percentile_year);
    END IF;
    INSERT OVERWRITE IDENTIFIER(:target_schema || '.openalex_works_metric_state')
    SELECT tonight, id, fwci, citation_normalized_percentile, cited_by_percentile_year FROM IDENTIFIER(:works_table);
  END IF;
END;

### Counts and consistency

Per change and per field, for the shadow week's report. Consistency counts go in the run row (all should be 0); they
are recorded, not raised, while the shadow week runs.

In [ ]:
DELETE FROM IDENTIFIER(:target_schema || '.openalex_works_field_change_counts') WHERE night = (SELECT max(night) FROM IDENTIFIER(:target_schema || '.openalex_works_field_runs'));

INSERT INTO IDENTIFIER(:target_schema || '.openalex_works_field_change_counts') (night, change, field, works)
SELECT night, change, CAST(NULL AS STRING), count(*) FROM IDENTIFIER(:target_schema || '.openalex_works_field_changes') WHERE night = (SELECT max(night) FROM IDENTIFIER(:target_schema || '.openalex_works_field_runs')) GROUP BY night, change
UNION ALL
SELECT night, change, f, count(*) FROM IDENTIFIER(:target_schema || '.openalex_works_field_changes') LATERAL VIEW explode(changed_fields) x AS f
WHERE night = (SELECT max(night) FROM IDENTIFIER(:target_schema || '.openalex_works_field_runs')) AND change = 'updated' GROUP BY night, change, f
UNION ALL
SELECT night, 'metric', CAST(NULL AS STRING), count(*) FROM IDENTIFIER(:target_schema || '.openalex_works_metric_changes') WHERE night = (SELECT max(night) FROM IDENTIFIER(:target_schema || '.openalex_works_field_runs')) GROUP BY night
UNION ALL
SELECT night, 'metric', f, count(*) FROM IDENTIFIER(:target_schema || '.openalex_works_metric_changes') LATERAL VIEW explode(changed_fields) x AS f
WHERE night = (SELECT max(night) FROM IDENTIFIER(:target_schema || '.openalex_works_field_runs')) GROUP BY night, f;

MERGE INTO IDENTIFIER(:target_schema || '.openalex_works_field_runs') AS t
USING (
  SELECT
    (SELECT max(night) FROM IDENTIFIER(:target_schema || '.openalex_works_field_runs')) AS night,
    count_if(w.id IS NOT NULL) AS works,
    count_if(s.id IS NULL) AS works_missing_from_state,
    count_if(w.id IS NULL) AS state_rows_not_in_works,
    count_if(w.id IS NOT NULL AND s.id IS NOT NULL AND NOT (w.updated_date <=> s.updated_date)) AS date_mismatches,
    count_if(s.id IS NOT NULL) - count(DISTINCT s.id) AS duplicate_ids
  FROM (SELECT id, updated_date FROM IDENTIFIER(:works_table)) w
  FULL OUTER JOIN (SELECT id, updated_date FROM IDENTIFIER(:target_schema || '.openalex_works_field_state')) s ON w.id = s.id
) AS c
ON t.night = c.night
WHEN MATCHED THEN UPDATE SET
  t.finished_at = current_timestamp(),
  t.works = c.works,
  t.works_missing_from_state = c.works_missing_from_state,
  t.state_rows_not_in_works = c.state_rows_not_in_works,
  t.date_mismatches = c.date_mismatches,
  t.duplicate_ids = c.duplicate_ids;